# SwiGLU Activation
*The gated feedforward activation in LLaMA, PaLM, Gemma, and most modern LLMs.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_vec4_load.py`


## What Is SwiGLU?

**The core idea: gated information flow.**

A standard activation function (ReLU, GELU) decides how much each neuron fires based only on its own value. SwiGLU adds a second input — a *gate* — that decides whether each feature is even worth passing forward at all. The gate is a learned selector: it can suppress entire feature directions that aren't useful for the current input.

Think of it as a highway with toll booths. The "up" projection builds candidate features (what information *could* pass). The "gate" projection controls the booths — it opens and closes them based on the input context. The model learns which booths to open for which kinds of inputs.

This gives the network finer-grained control over information flow than a simple activation. A single ReLU can zero out a neuron, but it can't suppress it selectively based on *what the input is*. SwiGLU's gate can.

**Why SwiGLU specifically?** The gate uses the SiLU (Sigmoid Linear Unit) function rather than a hard gate:

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

SiLU is smooth everywhere — no hard zero cutoff. This matters during training: even a gate that barely activates still has a nonzero gradient, so the model can learn to open that gate more. With ReLU, a gate that falls below zero is completely dead (gradient = 0) and can never recover.

**The empirical fact:** SwiGLU (Shazeer, 2020) consistently outperforms ReLU and GELU at the same parameter count — typically by 1–2 perplexity points on language modeling tasks. The theoretical reason isn't fully understood, but the pattern has held across model sizes, domains, and architectures. It's become the default in LLaMA, Mistral, Gemma, Qwen, and PaLM.

**Why it requires three weight matrices:**

Standard FFN: `Y = activation(X @ W_up.T) @ W_down.T` — two weights, one path.
SwiGLU FFN: `Y = (SiLU(X @ W_gate.T) × (X @ W_up.T)) @ W_down.T` — three weights, two parallel paths that get combined.

To keep total parameter count equivalent to a standard `4d` FFN, the intermediate dimension shrinks from `4d` to `8d/3 ≈ 2.67d`. At Qwen3-8B's `d=4096`: `8/3 × 4096 ≈ 10,923` → rounded up to **12,288** (the nearest multiple of 256). Three narrower matrices, same parameter budget.

**Why the gate is a learned dimmer, not a switch:**
- Large positive gate value → `SiLU(gate) ≈ gate` → feature passes through amplified
- Gate near zero → `SiLU(0) = 0` → feature suppressed regardless of up-projection
- Large negative gate → `SiLU(gate) ≈ 0` → feature effectively blocked

After training, the gate values for each input encode a kind of sparse selection: the model has learned which of the 12,288 intermediate feature channels are relevant for each kind of computation, and routes information accordingly.

In [ ]:
import math
print('ready')

## Building Blocks: SiLU and GLU

### SiLU (Sigmoid Linear Unit)

$$\text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

- When $x \gg 0$: $\sigma(x) \to 1$, so $\text{SiLU}(x) \approx x$ (identity — passes through)
- When $x \approx 0$: $\sigma(0) = 0.5$, so $\text{SiLU}(0) = 0$ (attenuated)
- When $x \ll 0$: $\sigma(x) \to 0$, so $\text{SiLU}(x) \approx 0$ (suppressed)

### GLU (Gated Linear Unit, Dauphin et al. 2017)

$$\text{GLU}(a, b) = a \otimes \sigma(b)$$

SwiGLU replaces $\sigma$ with SiLU:

$$\boxed{\text{SwiGLU}(x, y) = \text{SiLU}(x) \otimes y = \frac{x}{1+e^{-x}} \otimes y}$$


In [ ]:
def silu(x):
    return x / (1.0 + math.exp(-x))

def swiglu(gate, up):
    return silu(gate) * up

# Trace a single element
print("SiLU behavior:")
for x in [-3.0, -1.0, 0.0, 1.0, 3.0]:
    s = silu(x)
    print(f"  SiLU({x:+.1f}) = {s:+.4f}  "
          f"({'≈x' if abs(s-x)<0.1 else '≈0' if abs(s)<0.1 else ''})")

print()
# Element-wise example
gate = [2.0, -1.0, 0.5, 3.0]
up   = [1.5,  2.0, 0.8, 0.3]
print("SwiGLU element-wise:")
print(f"  gate:      {gate}")
print(f"  SiLU(gate): {[round(silu(g),3) for g in gate]}")
out = [swiglu(g, u) for g, u in zip(gate, up)]
print(f"  × up:      {[round(v,3) for v in out]}")


## The Formula

$$\boxed{\text{SwiGLU}(x, y) = \text{SiLU}(x) \cdot y = \frac{x}{1 + e^{-x}} \cdot y}$$

| Symbol | Meaning |
|--------|---------|
| $x$ | Gate vector (from one linear projection of the input) |
| $y$ | Up-projection vector (from another linear projection) |
| $\sigma(x) = \frac{1}{1+e^{-x}}$ | Sigmoid — squashes $x$ to $(0,1)$ |
| $\text{SiLU}(x) = x \cdot \sigma(x)$ | Smooth, differentiable, no "dead neuron" problem |
| $\otimes$ | Element-wise multiplication |

### 🗣️ Say It Out Loud
> *"SwiGLU of x and y equals x times sigma of x, times y — where sigma is the sigmoid function, one over one plus e to the minus x."*

**Tutor's gloss:** "Think of $x$ as the gate controlling a valve and $y$ as the flow through it. When the gate $x$ is large positive, $\sigma(x) \approx 1$, so the valve is fully open: $y$ passes through unchanged. When $x$ is near zero, the valve is half-open. When $x$ is large negative, it's nearly closed: $y$ is suppressed. The model learns to set the gate values to route information."


## SwiGLU in Context: The LLaMA FFN

A standard transformer FFN layer:
$$\text{FFN}(x) = W_\text{down} \cdot \text{ReLU}(W_\text{up} \cdot x)$$

LLaMA's SwiGLU FFN:
$$\text{FFN}_\text{SwiGLU}(x) = W_\text{down} \cdot \bigl(\text{SiLU}(W_\text{gate} \cdot x) \otimes W_\text{up} \cdot x\bigr)$$

Three weight matrices instead of two. To keep the same parameter count, LLaMA uses:

$$d_\text{hidden} = \left\lfloor \frac{8}{3} d_\text{model} \right\rfloor \approx 2.67 \times d_\text{model}$$

instead of the usual $4 \times d_\text{model}$.

| Variant | Activation | Weight matrices | Typical hidden dim |
|---------|-----------|-----------------|-------------------|
| Original Transformer | ReLU | 2 | $4d$ |
| GPT-2/3 | GELU | 2 | $4d$ |
| LLaMA / PaLM | SwiGLU | 3 | $8d/3 \approx 2.67d$ |

**Total parameters equivalent** (at $d=4096$): $2 \times 4096 \times 16384 = 2 \times 4096 \times 10923$ — same count.


## Optimization Ladder

| Version | Strategy | Key idea |
|---------|----------|----------|
| `v0_naive` | One thread per element, scalar loads | Establishes correctness baseline |
| `sm89_v1_vec4_load` | Vectorized `float4` loads | 4 BF16 values per load instruction — hits bandwidth ceiling |

**Why SwiGLU barely needs optimizing:**
The activation itself (SiLU × up) does ~3 FLOPs per element at 4 bytes in/out — barely above RMSNorm's ratio.
But it doesn't matter: the upstream GEMMs (W_gate·x and W_up·x) are 100–1000× more expensive.
The activation is always the cheapest part of the FFN forward pass.

## CuTeDSL: Fused Gate × Up Kernel

```python
@cute.kernel
def swiglu_kernel(mGate, mUp, mOut):
    # One thread per output element — no reduction needed
    i = blockIdx.x * blockDim.x + threadIdx.x

    gate = mGate[i]   # result of W_gate · x (one projection)
    up   = mUp[i]     # result of W_up · x (another projection)

    # SiLU(gate) = gate * sigmoid(gate) = gate / (1 + exp(-gate))
    # SM89 tip: cute.exp uses __expf, a fast 4-ulp approximation (~4× faster than full exp)
    silu_val = gate / (1.0 + cute.exp(-gate))

    mOut[i] = silu_val * up   # the "gate": controls how much of `up` passes through
```

**The v1 upgrade — loading 4 values at once:**
```python
# v0: one element per load
gate = mGate[i]    # 1 BF16 = 2 bytes per instruction

# v1: four elements per load  
gate_vec = mGate.load_v4(i)   # 4 BF16 = 8 bytes per instruction, same latency
# Then apply SiLU to each component of gate_vec
```
This directly increases the fraction of peak memory bandwidth actually achieved.

## RTX 4080: The Real Cost is in the Weight GEMMs, Not the Activation

The SwiGLU activation itself: ~3 FLOPs/element, 4 bytes/element → 0.75 FLOP/byte. Memory-bound.
But the activation takes microseconds. The weight GEMMs take milliseconds.

**Qwen3-8B MLP weight sizes:**
- W_gate: [12288, 4096] in BF16 = 100 MB
- W_up:   [12288, 4096] in BF16 = 100 MB
- W_down: [4096, 12288] in BF16 = 100 MB
- Total: 300 MB of weights per MLP layer, times 36 layers = 10.8 GB just for MLP weights

**At decode (M=1, generating one token):**
Reading W_gate + W_up + W_down each forward pass = 300 MB moved across HBM.
At 380 GB/s: 300 MB / 380 GB/s = **0.79 ms per decode step, just in weight reads**.
Tensor cores contribute almost nothing — the GPU waits on memory.

**FP8 quantization changes everything:**
- W_gate + W_up in FP8: 50 MB each (half the size)
- Reads drop from 300 MB → 150 MB per forward pass
- Time drops from 0.79 ms → ~0.40 ms
- SM89 native FP8 tensor cores give another ~2× on the compute side
- Total MLP speedup: roughly **2–3× faster decode** vs BF16

This is the biggest single optimization available on the RTX 4080 for inference: quantize the MLP weights to FP8.

## Suggestion: Fuse SwiGLU into the Down-Projection GEMM

The MLP forward pass currently launches three separate kernels:
```
1. gate_out = x @ W_gate.T   (GEMM)     shape: [M, 12288]  → HBM write: M×12288×2 bytes
2. up_out   = x @ W_up.T     (GEMM)     shape: [M, 12288]  → HBM write: M×12288×2 bytes
3. hidden   = SwiGLU(gate_out, up_out)  shape: [M, 12288]  → HBM read: 2 × M×12288×2
             (reads both gate_out and up_out)                 HBM write: M×12288×2 bytes
4. out      = hidden @ W_down.T (GEMM)  shape: [M, 4096]   → HBM read: M×12288×2 bytes
```

At **prefill (M=2048)**, the intermediate tensors ([2048, 12288] = 48 MB each) move through HBM four times: write gate_out, write up_out, read gate_out+up_out for SwiGLU, read hidden for down_proj.

**Fusion opportunity 1: fuse SwiGLU into gates kernel epilogue**
Instead of writing gate_out and up_out to HBM, compute them both in registers and immediately apply SwiGLU, writing only `hidden` to HBM. This requires running W_gate and W_up GEMMs with a shared thread block that accumulates both outputs simultaneously:

```python
# "Dual GEMM + fused activation" epilogue:
# Thread computes both gate_val and up_val for the same output position
rC_gate  = cute.partition_fragment_C(tiled_mma, ...)   # accumulator for gate
rC_up    = cute.partition_fragment_C(tiled_mma, ...)   # accumulator for up

for k_tile in range(K // Bk):
    # Load input tile (shared between both GEMMs — load once, use twice)
    cute.copy(mX_tile, sX)
    cute.sync_threads()

    cute.copy(mWgate_tile, sWgate)
    cute.gemm(tiled_mma, sX, sWgate, rC_gate)   # gate accumulation

    cute.copy(mWup_tile, sWup)
    cute.gemm(tiled_mma, sX, sWup,   rC_up)     # up accumulation

# Fused epilogue: SwiGLU before writing to HBM
for elem in range(elems_per_thread):
    gate = rC_gate[elem]
    up   = rC_up[elem]
    hidden = gate / (1.0 + cute.exp(-gate)) * up   # SwiGLU in registers
    mHidden[...] = hidden   # write to HBM only once
```

**Bytes saved at prefill M=2048:**
```
Without fusion: write gate (48 MB) + write up (48 MB) + read gate+up (96 MB) = 192 MB extra
With fusion:    write hidden only (48 MB)
Saved: 144 MB × 380 GB/s ≈ 0.38 ms per MLP layer per call
```

**Fusion opportunity 2: fuse hidden directly into W_down GEMM input**
Take it further: don't write `hidden` to HBM at all. Instead, stream SwiGLU output directly into the W_down GEMM as the A matrix. This is the "persistent kernel" approach used by FasterTransformer and TensorRT-LLM.

At prefill this is complex to implement. At decode (M=1), hidden is only 12,288 values (24 KB) — easily cached in L1, so the separate SwiGLU kernel barely matters.

## Decode Profiling Breakdown for One MLP Layer (Qwen3-8B)

At decode (one output token), here is where the time goes.
Estimates use 380 GB/s bandwidth and 57.5 TFLOPS BF16.

```
Operation           | Data read  | FLOPs      | BW-bound time | Compute-bound time
────────────────────────────────────────────────────────────────────────────────────
gate_proj (GEMM)    | 100 MB     | 33 MFLOPs  |   263 µs      |  0.6 µs
up_proj   (GEMM)    | 100 MB     | 33 MFLOPs  |   263 µs      |  0.6 µs
SwiGLU activation   |  72 MB     | 36 MFLOPs  |   190 µs      |  trivial
down_proj (GEMM)    | 100 MB     | 100 MFLOPs |   263 µs      |  1.7 µs
────────────────────────────────────────────────────────────────────────────────────
MLP total           | 372 MB     | 202 MFLOPs |  ~979 µs      |  ~3.5 µs
```

At decode: **memory-bound by 280×**. The compute time is a rounding error.

**The three levers to go faster:**

**1. Weight quantization (FP8):**
Cuts 300 MB of weights to 150 MB → ~490 µs per MLP layer (~2× faster).

**2. Larger batch size:**
At batch_size=8, all 8 tokens share the same weight reads.
The 300 MB is still read once, but now produces 8 outputs instead of 1.
Effective bandwidth per token: 300 MB ÷ 8 = 37.5 MB → ~99 µs per token.
Still memory-bound, but 8× more efficient.

**3. FP8 + batching together:**
150 MB weights ÷ 8 tokens → ~18 MB per token → approaching compute-bound territory.

**When does MLP become compute-bound?**
For down_proj at batch size M:
- FLOPs = 2 × M × 4096 × 12288
- Bytes ≈ M × 12288 × 2 (input) + 4096 × 12288 × 2 (weights) + M × 4096 × 2 (output)
- Crossover (intensity > 151 F/B) happens around **M ≈ 5**

Even at batch_size=5, the MLP is compute-bound. Weight quantization and batch size
are the two knobs worth reaching for on the RTX 4080.

## Where SwiGLU Lives in the Qwen3-8B Decode Step

SwiGLU is the activation in the MLP sublayer, steps ⑬–⑯ of each transformer layer.

```
┌──────────────────────────────────────────────────────────────────────────┐
│  Input: x [1, 4096]  ← normed hidden state from ln2 (post-attention)   │
│                                                                          │
│  ⑬ gate_proj   W[12288, 4096]  x @ W.T  → gate [1, 12288]  100 MB     │
│  ⑭ up_proj     W[12288, 4096]  x @ W.T  → up   [1, 12288]  100 MB     │
│                                                                          │
│  ⑮ SwiGLU:  hidden[i] = gate[i] / (1 + exp(-gate[i])) × up[i]         │
│     Elements processed: 12,288                                          │
│     FLOPs: ~3 per element × 12,288 = 36,864 total                      │
│     Compare: gate_proj GEMM = 2 × 1 × 12288 × 4096 = 100M FLOPs       │
│     SwiGLU activation = 0.04% of MLP compute                           │
│                                                                          │
│  ⑯ down_proj   W[4096, 12288]  hidden @ W.T  → [1, 4096]  100 MB      │
│  ⑰ Residual:   x = x + ffn_out  → [1, 4096]                           │
└──────────────────────────────────────────────────────────────────────────┘
```

### Why 12,288 and not 4 × 4096 = 16,384

Standard FFN: 2 matrices × [4096, 16384] = 2 × 67M params = 134M params per layer.
SwiGLU FFN: 3 matrices × [4096, X]. Matching parameter count: 3 × 4096 × X = 134M → X ≈ 10,923.
Rounded to nearest 256: **12,288**. Three matrices that sum to the same parameters as two wider ones.

### Decode MLP cost vs attention cost

```
MLP (steps ⑬–⑯) at decode T=4096:
  Read 300 MB weights + 48 KB activations → ~0.79 ms at 380 GB/s

Attention (step ⑨) at decode T=4096:
  Read 16 MB KV cache + 8 KB Q → ~0.04 ms at 380 GB/s

MLP is ~20× more expensive than attention at T=4096.
At T=32768: KV grows to 128 MB, but MLP weight read stays at 300 MB.
Even at very long context the MLP dominates decode latency.
```

### The gate as a learned sparse mask

Before training, gate values are random — some features pass, others don't. After training:
- Large positive gate → `SiLU(gate) ≈ gate` → feature passes through amplified
- Gate near zero → `SiLU(gate) ≈ 0` → feature suppressed
- Large negative gate → `SiLU(gate) ≈ 0` → feature gated off

The SiLU's smooth gradient (never exactly zero) means gates can recover from suppression during training — unlike ReLU, which has permanently dead neurons.

## SwiGLU Implementations in the Wild

### vLLM

`vllm/model_executor/layers/activation.py::SiluAndMul`
- Fused CUDA kernel: reads gate and up **concatenated** in one tensor `[1, 2×12288]`, applies SiLU to the first half, multiplies by the second half — all in one pass
- Avoids having two separate HBM allocations for `gate_out` and `up_out`
- Used automatically when the model config has `hidden_act = "silu"` and the MLP has `gate_proj` + `up_proj`

### FlashInfer

`flashinfer.activation.silu_and_mul(gate_up)` — same fused concatenated approach.
Input: `[1, 24576]` tensor (gate and up side by side).
Output: `[1, 12288]` after element-wise SiLU-gate.
Vectorized BF16 loads: 8 BF16 values per instruction (128-bit loads).

### Triton / Liger-Kernel

`liger_kernel.ops.swiglu` — Triton implementation:
```python
@triton.jit
def swiglu_kernel(gate_ptr, up_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    gate = tl.load(gate_ptr + offsets, mask=mask)
    up   = tl.load(up_ptr + offsets, mask=mask)
    silu = gate * tl.sigmoid(gate)
    tl.store(out_ptr + offsets, silu * up, mask=mask)
```
Straightforward, no reduction, good bandwidth utilization, ~10–20% slower than hand-tuned CUDA on SM89.

### TensorRT-LLM

SwiGLU is compiled into the MLP plugin, which handles `gate_proj + up_proj + SwiGLU + down_proj` as one unit. The intermediate `[1, 12288]` tensors for gate and up never touch HBM in the TRT execution plan.

### Dual GEMM + fused activation (the real optimization)

The maximum speedup comes from one kernel that:
1. Reads X once (shared input for both gate and up GEMMs)
2. Accumulates gate and up outputs in separate register tiles `rC_gate` and `rC_up`
3. Applies SwiGLU before writing anything to HBM
4. Writes only `hidden` to HBM (not `gate_out` or `up_out`)

This avoids writing `gate_out` (100 MB at prefill) and `up_out` (100 MB) to HBM entirely — a 200 MB bandwidth saving per MLP layer. FasterTransformer and TRT-LLM both implement this pattern. It is the next logical step after the standalone `sm89_v1_vec4_load` kernel in this project.

## One-Sentence Takeaway

SwiGLU is the most expensive operation in Qwen3-8B inference not because the activation itself is costly (36K FLOPs, done in microseconds) but because it sits between two of the three largest weight matrices in the model (gate_proj and up_proj at 100 MB each in BF16), making the full MLP block 20× more expensive than attention at typical decode context lengths — so the optimization target is never the SiLU computation but always the surrounding weight reads, where FP8 quantization halves the 300 MB per-layer footprint and fused dual-GEMM epilogues eliminate the intermediate HBM writes for gate and up, with the SwiGLU gating mechanism itself being simply the learned control signal by which the model decides which of the 12,288 intermediate features to amplify or suppress.

## Community Optimization Resources for SwiGLU

### Measured speedups on SM89-class hardware

| Source | Impl | HW | Result | Notes |
|---|---|---|---|---|
| [qwen3-tts-triton](https://github.com/newgrit1004/qwen3-tts-triton) | Triton | RTX 5090 | HBM trips **3 → 1** | Gate + SiLU activation fused; eliminates `gate_out` and `up_out` intermediate writes |
| [qwen3.5-triton](https://github.com/RightNow-AI/qwen3.5-triton) | Triton | B200 | Confirmed fused | SiLU MLP gating + multiplication fused into single kernel |
| [qwen3-tts-triton](https://github.com/newgrit1004/qwen3-tts-triton) | Triton | **RTX 4090 (SM89)** | **5.16×** system | Triton + CUDA graphs + static KV; SwiGLU fusion is part of the kernel stack |

### Implementations to study

| Repo / file | Impl | What it shows |
|---|---|---|
| [qwen3-tts-triton fused SwiGLU](https://github.com/newgrit1004/qwen3-tts-triton) | Triton | Single Triton kernel: reads gate and up tensors, applies `silu(gate) * up`, writes hidden — 1 HBM write instead of 3 passes |
| [qwen3.5-triton](https://github.com/RightNow-AI/qwen3.5-triton) | Triton | Profiler: cuBLAS is 80.4% of decode time; SwiGLU activation is a rounding error |

### HBM traffic comparison at M=2048 prefill (Qwen3-8B, BF16)

```
gate_out and up_out shapes: [2048, 12288] = 48 MB each

Unfused (3 passes):
  GEMM W_gate: writes 48 MB gate_out
  GEMM W_up:   writes 48 MB up_out
  SwiGLU:      reads 96 MB (gate_out + up_out), writes 48 MB hidden
  = 240 MB total HBM traffic for the activation stage

Triton fused (HBM 3 → 1):
  SwiGLU kernel: reads gate_out + up_out (96 MB), writes hidden (48 MB)
  = saves 2 × 48 MB = 96 MB vs unfused
  (gate_out and up_out are still written to HBM by the two separate GEMMs)

CuTeDSL dual-accumulator epilogue (HBM 0 writes for gate/up):
  GEMM loop holds rC_gate and rC_up as separate register accumulators
  SwiGLU applied in-register before any HBM write
  Only hidden is written (48 MB total for the activation stage)
  = saves 96 MB vs Triton fusion, 192 MB vs unfused
  = ~0.25 ms additional savings vs Triton at 380 GB/s
```

### CuTeDSL dual-accumulator pattern (unique to CuTeDSL — Triton cannot do this)

```python
rC_gate = zeros_like_accum()    # register accumulator for W_gate output
rC_up   = zeros_like_accum()    # register accumulator for W_up output

for k_tile in cute.range(K // BK):
    cute.copy(mX_tile, sX)           # load input tile ONCE — shared by both
    cute.copy(mWgate_tile, sWgate)
    cute.copy(mWup_tile, sWup)
    cute.gemm(tiled_mma, sX, sWgate, rC_gate)   # accumulate gate
    cute.gemm(tiled_mma, sX, sWup,   rC_up)     # accumulate up

# Epilogue: SwiGLU in-register, one HBM write
for i in cute.range(elems_per_thread):
    g = rC_gate[i]
    hidden_reg = g / (1.0 + cute.exp(-g, fastmath=True)) * rC_up[i]
    mHidden[...] = hidden_reg       # gate_out and up_out NEVER touch HBM
```

Triton cannot hold two independent MMA accumulators simultaneously in the same
kernel — you would need to write gate_out to HBM, launch a second GEMM for up_out,
then fuse the activation. CuTeDSL's explicit register management enables the
zero-intermediate-write pattern.

### What dominates MLP time at decode

The activation itself is microseconds. The weight reads dominate:

```
Operation           Data read   Time @ 380 GB/s
────────────────────────────────────────────────
gate_proj (GEMM)     100 MB      0.26 ms
up_proj   (GEMM)     100 MB      0.26 ms
SwiGLU activation     ~0         ~0 (negligible)
down_proj (GEMM)     100 MB      0.26 ms
────────────────────────────────────────────────
One MLP layer        300 MB      0.79 ms
36 layers           10.8 GB     ~28.4 ms/token
```

Weight quantization (FP8) and batch size are the two biggest levers for MLP decode.
The SwiGLU fusion optimization matters most at **prefill** (large M), where intermediate
tensors are large enough to measure.

### Recommended implementation order

1. **Triton fused SwiGLU** — single kernel, reads gate_out + up_out, writes hidden (HBM 3→1)
2. **CuTeDSL dual-accumulator GEMM epilogue** — no intermediate HBM writes for gate/up (HBM 0)
3. **CUDA graphs** — eliminate kernel launch overhead for the 3-GEMM + activation sequence